In [5]:
from __future__ import annotations

import sys
import subprocess
from pathlib import Path


def ensure(pkg: str) -> None:
    try:
        __import__(pkg)
        return
    except Exception:
        pass

    print(f"Installing: {pkg} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])


# RDKit (pip版)
# 失敗する環境もあるので、その場合はエラー内容を表示します。
try:
    import rdkit  # noqa: F401
except Exception:
    ensure("rdkit-pypi")

try:
    import pandas as pd  # noqa: F401
except Exception:
    ensure("pandas")

print("OK: imports ready")
print("Working dir:", Path.cwd())

OK: imports ready
Working dir: /home/yamamoto/Work/chemistry/orca-work/MEA


In [6]:
from rdkit import Chem
from rdkit.Chem import AllChem

import pandas as pd

smiles = "NCCO"
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)

# 3Dコンフォマー生成
num_confs = 50
params = AllChem.ETKDGv3()
params.randomSeed = 42
conf_ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=num_confs, params=params))
print("Embedded conformers:", len(conf_ids))

# MMFF最適化 & エネルギー算出
mmff_props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant="MMFF94")
energies = []
for cid in conf_ids:
    ff = AllChem.MMFFGetMoleculeForceField(mol, mmff_props, confId=cid)
    ff.Minimize(maxIts=500)
    e = float(ff.CalcEnergy())
    energies.append((cid, e))

energies_sorted = sorted(energies, key=lambda x: x[1])
e0 = energies_sorted[0][1]
rows = []
for rank, (cid, e) in enumerate(energies_sorted, start=1):
    rows.append({"rank": rank, "confId": cid, "E_MMFF": e, "dE": e - e0})

df = pd.DataFrame(rows)
df.head(10)

Embedded conformers: 50


,rank,confId,E_MMFF,dE
0,1,43,13.782177,0.000000e+00
1,2,32,13.782177,2.375256e-10
2,3,28,13.782177,1.319474e-09
3,4,39,13.782177,1.401387e-09
4,5,49,13.782177,2.127980e-09
5,6,27,13.782177,2.199329e-09
6,7,29,13.782177,2.742629e-09
7,8,6,13.782177,2.758446e-09
8,9,34,13.782177,3.357874e-09
9,10,33,13.782177,5.097537e-09


In [7]:
# 上位コンフォマーをXYZ保存
out_dir = Path("conformers")
out_dir.mkdir(parents=True, exist_ok=True)

top_n = 10
top = df.head(top_n)

for _, r in top.iterrows():
    cid = int(r["confId"])
    rank = int(r["rank"])
    dE = float(r["dE"])

    block = Chem.MolToXYZBlock(mol, confId=cid)
    lines = block.splitlines()
    if len(lines) >= 2:
        # 1行目(原子数)は維持し、2行目(コメント)のみ差し替える
        lines[1] = f"MEA conformer rank={rank} dE(MMFF)={dE:.6f}"
        xyz = "\n".join(lines) + "\n"
    else:
        # 念のため(通常ここには来ない)
        xyz = f"{block.rstrip()}\n"

    out_path = out_dir / f"mea_conf_{rank:02d}.xyz"
    out_path.write_text(xyz)

print("Saved:", top_n, "structures to", out_dir.resolve())
top

Saved: 10 structures to /home/yamamoto/Work/chemistry/orca-work/MEA/conformers


,rank,confId,E_MMFF,dE
0,1,43,13.782177,0.000000e+00
1,2,32,13.782177,2.375256e-10
2,3,28,13.782177,1.319474e-09
3,4,39,13.782177,1.401387e-09
4,5,49,13.782177,2.127980e-09
5,6,27,13.782177,2.199329e-09
6,7,29,13.782177,2.742629e-09
7,8,6,13.782177,2.758446e-09
8,9,34,13.782177,3.357874e-09
9,10,33,13.782177,5.097537e-09
